# Normalization Tradeoffs

This notebook extends the general FD walkthrough with small, executable teaching checks for BCNF decomposition, lossless joins, dependency preservation, and a toy storage estimate.

The goal is still exploratory: make the definitions concrete on small schemas, not build a production normal-form optimizer.

## Setup

In [7]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))
for module_name in tuple(sys.modules):
    if module_name == "fdkeys" or module_name.startswith("fdkeys."):
        del sys.modules[module_name]

from fdkeys import (
    FD,
    closure,
    decompose_bcnf,
    estimate_storage_bytes,
    explain_3nf_synthesis,
    format_fd,
    format_relation,
    is_lossless_binary_decomposition,
    preserves_dependencies,
    project_fds,
    relation_width,
    synthesize_3nf,
)

## 1. A Classic Tradeoff Example

Use a small general-FD schema where BCNF decomposition is lossless, but dependency preservation can fail.

- `AB -> C`: `A` and `B` are needed together.
- `C -> B`: `C` determines `B`.

The candidate keys are `AB` and `AC`. The FD `C -> B` violates BCNF because `C` is not a superkey for the whole relation.

In [8]:
universe = frozenset({"A", "B", "C"})
fds = [
    FD(frozenset({"A", "B"}), "C"),
    FD(frozenset({"C"}), "B"),
]

print("Universe:", format_relation(universe))
print("FDs:")
for fd in fds:
    print(" ", format_fd(fd))
print("closure(AB):", format_relation(closure({"A", "B"}, fds)))
print("closure(AC):", format_relation(closure({"A", "C"}, fds)))
print("closure(C):", format_relation(closure({"C"}, fds)))

Universe: {A, B, C}
FDs:
  {A, B} -> C
  {C} -> B
closure(AB): {A, B, C}
closure(AC): {A, B, C}
closure(C): {B, C}


## 2. 3NF Synthesis

The compact 3NF synthesis keeps the dependencies easy to enforce. In this example, the synthesized relation containing `ABC` subsumes the smaller relation that would come from `C -> B`.

In [9]:
trace = explain_3nf_synthesis(universe, fds)
three_nf = synthesize_3nf(universe, fds)

print("Minimal cover:")
for fd in trace.minimal_cover:
    print(" ", format_fd(fd))

print("\nGenerated relations:")
for step in trace.generated_relations:
    print(" ", format_relation(step.relation), "from", format_relation(step.determinant))

print("\nRemoved subsumed:", [format_relation(rel) for rel in trace.removed_subsumed_relations])
print("Final 3NF:", [format_relation(rel) for rel in three_nf])
print("Dependency-preserving?", preserves_dependencies(universe, fds, three_nf))

Minimal cover:
  {A, B} -> C
  {C} -> B

Generated relations:
  {A, B, C} from {A, B}
  {B, C} from {C}

Removed subsumed: ['{B, C}']
Final 3NF: ['{A, B, C}']
Dependency-preserving? True


## 3. BCNF Decomposition

BCNF recursively splits on a dependency whose left-hand side is not a superkey. Here, splitting on `C -> B` gives `{B, C}` and `{A, C}`.

In [10]:
bcnf = decompose_bcnf(universe, fds)
print("BCNF decomposition:", [format_relation(rel) for rel in bcnf])

left, right = bcnf
print("Lossless binary decomposition?", is_lossless_binary_decomposition(universe, fds, left, right))
print("Dependency-preserving?", preserves_dependencies(universe, fds, bcnf))

BCNF decomposition: ['{A, C}', '{B, C}']
Lossless binary decomposition? True
Dependency-preserving? False


## 4. Why Preservation Fails Here

Dependency preservation asks whether the projected dependencies from each decomposed relation can derive the original dependencies without joining tables back together.

After the BCNF split, `C -> B` is visible inside `{B, C}`, but `AB -> C` is not derivable from the union of projected dependencies.

In [11]:
projected = []
for relation in bcnf:
    relation_projection = project_fds(universe, fds, relation)
    projected.extend(relation_projection)
    print(format_relation(relation), "projection:")
    for fd in relation_projection:
        print(" ", format_fd(fd))

print("\nclosure(AB) under projected FDs:", format_relation(closure({"A", "B"}, projected)))

{A, C} projection:
{B, C} projection:
  {C} -> B

closure(AB) under projected FDs: {A, B}


## 5. Toy Storage Estimate

A normal form algorithm does not know real row counts or attribute widths. Still, a teaching model can make the tradeoff concrete by accepting estimated widths and estimated relation cardinalities.

These numbers are illustrative only. A real database design would need workload, keys, indexes, nullability, compression, and measured cardinalities.

In [12]:
attribute_widths = {"A": 8, "B": 32, "C": 120}

original = [universe]
original_counts = {universe: 1_000_000}
three_nf_counts = {relation: 1_000_000 for relation in three_nf}
bcnf_counts = {
    frozenset({"A", "C"}): 1_000_000,
    frozenset({"B", "C"}): 50_000,
}

for relation in [universe, *bcnf]:
    print(format_relation(relation), "tuple width:", relation_width(relation, attribute_widths))

print("\nEstimated bytes:")
print(" original:", estimate_storage_bytes(original, original_counts, attribute_widths))
print(" 3NF:", estimate_storage_bytes(three_nf, three_nf_counts, attribute_widths))
print(" BCNF:", estimate_storage_bytes(bcnf, bcnf_counts, attribute_widths))

{A, B, C} tuple width: 160
{A, C} tuple width: 128
{B, C} tuple width: 152

Estimated bytes:
 original: 160000000
 3NF: 160000000
 BCNF: 135600000


## 6. Takeaways

- 3NF synthesis is dependency-preserving by construction for the cover used here.
- BCNF gives a stricter normal form, and the binary split is lossless in this example.
- BCNF can lose dependency preservation, so checking `AB -> C` may require a join after decomposition.
- Storage-aware reasoning needs extra estimates. Candidate keys and FDs help, but they are not enough by themselves to predict space usage exactly.
- The code in this notebook is intentionally exhaustive and small-schema oriented.